# Notebook 02 — Model Building

Trains 5 classifiers, evaluates on the held-out test set, plots confusion
matrices & ROC curves, and saves all models to `../models/`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')

from src.preprocessing import load_preprocessed
from src.model_utils import train_model, evaluate_model, save_model
from src.evaluation import (
    plot_confusion_matrix, plot_roc_curve,
    classification_summary, compare_models_table
)

In [2]:
X_train, X_test, y_train, y_test, scaler = load_preprocessed('../outputs')
print(f'Features ({X_train.shape[1]}): {list(X_train.columns)}')
print(f'Train class dist:\n{y_train.value_counts()}')
print(f'Test  class dist:\n{y_test.value_counts()}')

Loaded  X_train=(16296, 14)  X_test=(2474, 14)
Features (14): ['age', 'income', 'assets', 'credit_score', 'debt_to_income_ratio', 'existing_loan', 'criminal_record', 'credit_tier', 'high_debt', 'asset_income_ratio', 'age_group', 'income_per_age', 'payment_capacity', 'credit_utilization_proxy']
Train class dist:
loan_approved
0    8148
1    8148
Name: count, dtype: int64
Test  class dist:
loan_approved
0    2037
1     437
Name: count, dtype: int64


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

models = {
    'Logistic_Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random_Forest': RandomForestClassifier(n_estimators=200, max_depth=15,
                                            min_samples_split=5, min_samples_leaf=2,
                                            random_state=42, n_jobs=-1),
    'Gradient_Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                                    learning_rate=0.1, subsample=0.8,
                                                    random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=42, eval_metric='logloss',
                             use_label_encoder=False),
    'SVM': SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
}
print(f'Models to train: {list(models.keys())}')

Models to train: ['Logistic_Regression', 'Random_Forest', 'Gradient_Boosting', 'XGBoost', 'SVM']


In [4]:
all_results = []
trained = {}

for name, mdl in models.items():
    m = train_model(mdl, X_train, y_train, model_name=name)
    trained[name] = m
    metrics, y_pred, y_prob = evaluate_model(m, X_test, y_test, model_name=name)
    all_results.append(metrics)
    classification_summary(y_test, y_pred, model_name=name)

Training Logistic_Regression ... 

done in 0.02s
  ✓ Sanity checks passed for Logistic_Regression

Classification Report - Logistic_Regression
              precision    recall  f1-score   support

    Rejected       0.90      0.63      0.74      2037
    Approved       0.28      0.68      0.40       437

    accuracy                           0.64      2474
   macro avg       0.59      0.65      0.57      2474
weighted avg       0.79      0.64      0.68      2474

Training Random_Forest ... 

done in 1.58s
  ✓ Sanity checks passed for Random_Forest

Classification Report - Random_Forest
              precision    recall  f1-score   support

    Rejected       0.89      0.89      0.89      2037
    Approved       0.49      0.48      0.49       437

    accuracy                           0.82      2474
   macro avg       0.69      0.69      0.69      2474
weighted avg       0.82      0.82      0.82      2474

Training Gradient_Boosting ... 

done in 8.74s
  ✓ Sanity checks passed for Gradient_Boosting

Classification Report - Gradient_Boosting
              precision    recall  f1-score   support

    Rejected       0.88      0.93      0.91      2037
    Approved       0.57      0.42      0.48       437

    accuracy                           0.84      2474
   macro avg       0.73      0.68      0.70      2474
weighted avg       0.83      0.84      0.83      2474

Training XGBoost ... 

done in 0.35s
  ✓ Sanity checks passed for XGBoost

Classification Report - XGBoost
              precision    recall  f1-score   support

    Rejected       0.88      0.92      0.90      2037
    Approved       0.52      0.41      0.46       437

    accuracy                           0.83      2474
   macro avg       0.70      0.67      0.68      2474
weighted avg       0.82      0.83      0.82      2474

Training SVM ... 

done in 20.06s


  ✓ Sanity checks passed for SVM

Classification Report - SVM
              precision    recall  f1-score   support

    Rejected       0.90      0.77      0.83      2037
    Approved       0.37      0.61      0.46       437

    accuracy                           0.74      2474
   macro avg       0.63      0.69      0.65      2474
weighted avg       0.81      0.74      0.77      2474



In [5]:
comp = compare_models_table(all_results)
print('\nMODEL COMPARISON (sorted by F1)\n')
print(comp.to_string(index=False))


MODEL COMPARISON (sorted by F1)

              Model Accuracy Precision Recall F1_Score ROC_AUC
      Random_Forest   0.8205    0.4918 0.4828   0.4873  0.7158
  Gradient_Boosting   0.8420    0.5714 0.4211   0.4848  0.7035
            XGBoost   0.8286    0.5186 0.4142   0.4606  0.7021
                SVM   0.7445    0.3659 0.6087   0.4570  0.7089
Logistic_Regression   0.6399    0.2826 0.6751   0.3984  0.7269


In [6]:
# Confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.ravel()
for i, (name, mdl) in enumerate(trained.items()):
    y_pred = mdl.predict(X_test)
    plot_confusion_matrix(y_test, y_pred, model_name=name, ax=axes[i])
axes[-1].axis('off')
plt.suptitle('Confusion Matrices', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [7]:
# ROC curves
fig, ax = plt.subplots(figsize=(9, 7))
for name, mdl in trained.items():
    y_prob = mdl.predict_proba(X_test)[:, 1]
    plot_roc_curve(y_test, y_prob, model_name=name, ax=ax)
ax.set_title('ROC Curves - All Models', fontweight='bold', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
plt.tight_layout()
plt.savefig('../outputs/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [8]:
# Feature importance (tree models)
tree_names = ['Random_Forest', 'Gradient_Boosting', 'XGBoost']
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for i, name in enumerate(tree_names):
    imp = trained[name].feature_importances_
    idx = np.argsort(imp)
    axes[i].barh(range(len(idx)), imp[idx], color='steelblue', edgecolor='black')
    axes[i].set_yticks(range(len(idx)))
    axes[i].set_yticklabels(X_train.columns[idx], fontsize=9)
    axes[i].set_title(name, fontweight='bold')
plt.suptitle('Feature Importance', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [9]:
# Cross-validation
from sklearn.model_selection import cross_val_score
print(f'{"Model":<25} {"CV Acc":>10} {"CV F1":>10} {"Std":>10}')
print('-'*60)
for name, mdl in trained.items():
    cv_f1 = cross_val_score(mdl, X_train, y_train, cv=5, scoring='f1')
    cv_acc = cross_val_score(mdl, X_train, y_train, cv=5, scoring='accuracy')
    print(f'{name:<25} {cv_acc.mean():>10.4f} {cv_f1.mean():>10.4f} {cv_f1.std():>10.4f}')

Model                         CV Acc      CV F1        Std
------------------------------------------------------------
Logistic_Regression           0.6553     0.6603     0.0108


Random_Forest                 0.8788     0.8734     0.0714


Gradient_Boosting             0.8695     0.8323     0.1975


XGBoost                       0.8606     0.7106     0.1863


SVM                           0.7185     0.7016     0.0062


In [10]:
# Save all models
for name, mdl in trained.items():
    save_model(mdl, name, model_dir='../models')

# Save comparison CSV
pd.DataFrame(all_results).to_csv('../outputs/model_comparison.csv', index=False)
print('\nAll models & results saved.')

Saved ../models/Logistic_Regression.pkl
Saved ../models/Random_Forest.pkl
Saved ../models/Gradient_Boosting.pkl
Saved ../models/XGBoost.pkl
Saved ../models/SVM.pkl

All models & results saved.
